In [1]:
from pathlib import Path
import pandas as pd


# =========================
# 1. 文件路径
# =========================

INPUT_FILE = Path("/Users/tanglikun/Desktop/A股AI产业链召回总表_第一轮.xlsx")
OUTPUT_FILE = Path("/Users/tanglikun/Desktop/A股AI产业链召回总表_第二轮.xlsx")


# =========================
# 2. 第二层 AI 产业链 recall 关键词
# =========================

SECOND_LAYER_KEYWORDS = {
    "AI基础层-算力硬件": [
        "服务器", "AI服务器", "智算中心", "智能算力中心", "算力中心",
        "数据中心", "IDC", "云数据中心", "超算中心", "边缘计算",
        "云计算", "云服务", "云平台", "云基础设施"
    ],
    "AI基础层-芯片半导体": [
        "芯片", "AI芯片", "GPU", "GPGPU", "ASIC", "FPGA",
        "SoC", "NPU", "DPU", "TPU", "推理芯片", "训练芯片",
        "车规芯片", "存储芯片", "DRAM", "NAND", "HBM",
        "先进封装", "Chiplet", "封装基板", "半导体设备",
        "半导体材料", "EDA", "晶圆", "光刻", "刻蚀"
    ],
    "AI基础层-通信与连接": [
        "光模块", "高速光模块", "800G", "1.6T", "CPO", "硅光",
        "光芯片", "光器件", "光通信", "光互联", "交换机",
        "路由器", "网络设备", "以太网", "InfiniBand",
        "连接器", "高速连接器", "线缆", "高速线缆"
    ],
    "AI基础层-PCB与散热": [
        "PCB", "高速PCB", "HDI", "载板", "封装载板",
        "散热", "液冷", "冷板液冷", "浸没式液冷", "液冷服务器",
        "温控", "机柜", "电源模块", "UPS", "电源管理"
    ],
    "AI模型与数据层": [
        "数据标注", "数据清洗", "数据治理", "数据要素", "训练数据",
        "语料", "知识图谱", "向量数据库", "数据库", "中间件",
        "模型训练", "模型推理", "推理引擎", "算法平台",
        "AI平台", "MaaS", "机器学习平台", "深度学习框架"
    ],
    "AI感知层": [
        "传感器", "图像传感器", "视觉传感器", "摄像头模组",
        "激光雷达", "毫米波雷达", "超声波雷达", "红外传感器",
        "MEMS", "机器视觉", "工业视觉", "视觉检测", "缺陷检测"
    ],
    "AI应用-机器人": [
        "机器人", "工业机器人", "服务机器人", "人形机器人",
        "协作机器人", "移动机器人", "AMR", "AGV", "机器人控制器",
        "伺服系统", "减速器", "机器视觉", "机器人本体",
        "机器人关节", "灵巧手"
    ],
    "AI应用-智能汽车": [
        "自动驾驶", "智能驾驶", "辅助驾驶", "ADAS", "智能座舱",
        "车路协同", "智能网联", "车载计算", "域控制器",
        "驾驶域控", "座舱域控", "线控底盘", "高精地图",
        "智能汽车", "无人驾驶"
    ],
    "AI应用-行业场景": [
        "智能制造", "工业互联网", "智慧工厂", "智慧矿山",
        "智慧医疗", "医学影像", "智能诊断", "AI制药",
        "智慧金融", "智能客服", "智慧城市", "智能安防",
        "智慧交通", "智慧能源", "智慧电网", "智慧物流",
        "智能仓储", "智慧教育", "智慧政务"
    ],
    "AI应用-AIGC生态": [
        "数字人", "虚拟人", "虚拟主播", "AIGC", "内容生成",
        "智能创作", "AI绘画", "AI视频", "AI办公", "智能办公",
        "智能营销", "智能投顾", "OCR", "RPA"
    ],
    "AI终端与交互": [
        "智能终端", "智能穿戴", "智能家居", "AR", "VR", "MR",
        "XR", "空间计算", "智能音箱", "智能语音", "语音交互",
        "无人机"
    ],
}


# =========================
# 3. 容易误伤的弱词
# 命中这些不直接判“是”，只判“不能确定”
# =========================

WEAK_KEYWORDS = [
    "智能", "数字化", "自动化", "信息化", "平台化", "云平台",
    "数据平台", "解决方案", "系统集成"
]


# =========================
# 4. 工具函数
# =========================

def read_table(path):
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, dtype=str)
    return pd.read_csv(path, dtype=str, encoding="utf-8-sig")


def find_hits_by_layer(text, keyword_dict):
    text = str(text)
    hit_keywords = []
    hit_layers = []

    for layer, keywords in keyword_dict.items():
        for kw in keywords:
            if kw in text:
                hit_keywords.append(kw)
                hit_layers.append(layer)

    hit_keywords = list(dict.fromkeys(hit_keywords))
    hit_layers = list(dict.fromkeys(hit_layers))
    return hit_keywords, hit_layers


def find_hits(text, keywords):
    text = str(text)
    return [kw for kw in keywords if kw in text]


def get_hit_text(text, hits, max_len=220):
    if not hits:
        return ""

    text = str(text).replace("\n", " ").replace("\r", " ")

    for kw in hits:
        pos = text.find(kw)
        if pos != -1:
            start = max(pos - 80, 0)
            end = min(pos + 140, len(text))
            return text[start:end][:max_len]

    return ""


def append_text(old, new):
    old = "" if pd.isna(old) else str(old)
    new = "" if pd.isna(new) else str(new)

    if not old:
        return new
    if not new:
        return old
    return old + "；" + new


# =========================
# 5. 主程序
# =========================

df = read_table(INPUT_FILE)
df = df.fillna("")

# 如果第一轮文件里没有这些列，就先补上
needed_cols = [
    "是否在AI产业链中",
    "召回来源",
    "命中关键词",
    "命中文本",
    "AI产业链环节",
    "确定性等级",
    "复核状态",
    "备注",
]

for col in needed_cols:
    if col not in df.columns:
        df[col] = ""

# 用每一行所有文字字段做搜索
search_text = df.astype(str).agg(" ".join, axis=1)

step2_sources = []
step2_keywords = []
step2_texts = []
step2_layers = []

for i, text in enumerate(search_text):
    current_judgment = df.loc[i, "是否在AI产业链中"]

    strong_hits, layers = find_hits_by_layer(text, SECOND_LAYER_KEYWORDS)
    weak_hits = find_hits(text, WEAK_KEYWORDS)

    step2_keywords.append("、".join(strong_hits + weak_hits))
    step2_texts.append(get_hit_text(text, strong_hits + weak_hits))
    step2_layers.append("、".join(layers))

    if strong_hits:
        step2_sources.append("第二层产业链关键词召回")

        # 如果第一轮已经是“是”，保留“是”，并补充证据
        df.loc[i, "是否在AI产业链中"] = "是"
        df.loc[i, "召回来源"] = append_text(df.loc[i, "召回来源"], "第二层产业链关键词召回")
        df.loc[i, "命中关键词"] = append_text(df.loc[i, "命中关键词"], "、".join(strong_hits))
        df.loc[i, "命中文本"] = append_text(df.loc[i, "命中文本"], get_hit_text(text, strong_hits))
        df.loc[i, "AI产业链环节"] = append_text(df.loc[i, "AI产业链环节"], "、".join(layers))
        df.loc[i, "确定性等级"] = "中"
        df.loc[i, "复核状态"] = "待复核"
        df.loc[i, "备注"] = append_text(
            df.loc[i, "备注"],
            "第二层命中AI产业链上下游关键词，纳入候选池，后续用年报/主营业务/经营范围复核"
        )

    elif weak_hits:
        step2_sources.append("第二层弱相关关键词命中")

        # 只有原来不是“是”的，才改成不能确定
        if current_judgment != "是":
            df.loc[i, "是否在AI产业链中"] = "不能确定"
            df.loc[i, "召回来源"] = append_text(df.loc[i, "召回来源"], "第二层弱相关关键词命中")
            df.loc[i, "命中关键词"] = append_text(df.loc[i, "命中关键词"], "、".join(weak_hits))
            df.loc[i, "命中文本"] = append_text(df.loc[i, "命中文本"], get_hit_text(text, weak_hits))
            df.loc[i, "确定性等级"] = "低"
            df.loc[i, "复核状态"] = "进入下一轮"
            df.loc[i, "备注"] = append_text(
                df.loc[i, "备注"],
                "仅命中智能/数字化/自动化等弱相关词，不能直接确认，需要下一轮文本或年报复核"
            )

    else:
        step2_sources.append("第二轮未召回")

        # 原来是不能确定的继续不能确定，原来是否就保持否
        if current_judgment == "不能确定":
            df.loc[i, "复核状态"] = "进入下一轮"


# 单独保留第二轮的信息，方便你检查
df["第二层召回来源"] = step2_sources
df["第二层命中关键词"] = step2_keywords
df["第二层命中文本"] = step2_texts
df["第二层AI产业链环节"] = step2_layers


# 保存
df.to_excel(OUTPUT_FILE, index=False)

print("第二轮 recall 完成！")
print(f"输出文件：{OUTPUT_FILE}")
print()
print("当前总判断结果：")
print(df["是否在AI产业链中"].value_counts())
print()
print("第二层召回情况：")
print(pd.Series(step2_sources).value_counts())

第二轮 recall 完成！
输出文件：/Users/tanglikun/Desktop/A股AI产业链召回总表_第二轮.xlsx

当前总判断结果：
是否在AI产业链中
不能确定    5370
否        149
是          8
Name: count, dtype: int64

第二层召回情况：
第二轮未召回         5439
第二层弱相关关键词命中      80
第二层产业链关键词召回       8
Name: count, dtype: int64
